<a href="https://www.kaggle.com/code/noshabafatima/aadhaar-demographic-analysis?scriptVersionId=323679171" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Aadhaar Enrollment Analysis & Anomaly Detection

## Problem Statement
The objective of this project is to identify meaningful trends, anomalies, and predictive indicators in Aadhaar enrollment data and translate them into actionable insights and solution frameworks to support informed decision-making and system improvement.

## Dataset Overview
The dataset contains Aadhaar enrollment statistics aggregated at district, state, and national levels across different time periods and age groups.


## Environment Setup
This section imports the required Python libraries for data manipulation, analysis, and visualization.


In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Integration
Multiple CSV files are loaded and merged to create a unified Aadhaar enrollment dataset for analysis.


In [ ]:
import pandas as pd

df1 = pd.read_csv('/kaggle/input/aadhaar-demographic-dataset/api_data_aadhar_demographic/api_data_aadhar_demographic_0_500000.csv')
df2 = pd.read_csv('/kaggle/input/aadhaar-demographic-dataset/api_data_aadhar_demographic/api_data_aadhar_demographic_1000000_1500000.csv')
df3 = pd.read_csv('/kaggle/input/aadhaar-demographic-dataset/api_data_aadhar_demographic/api_data_aadhar_demographic_1500000_2000000.csv')
df4 = pd.read_csv('/kaggle/input/aadhaar-demographic-dataset/api_data_aadhar_demographic/api_data_aadhar_demographic_2000000_2071700.csv')
df5 = pd.read_csv('/kaggle/input/aadhaar-demographic-dataset/api_data_aadhar_demographic/api_data_aadhar_demographic_500000_1000000.csv')


In [ ]:
df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)


## Initial Data Exploration:
 Dataset Structure
 
 Data Types Verification
 
 Descriptive Statistics



In [ ]:
df.head()
df.shape
df.tail()
print(df.columns)
print(df.describe())



In [ ]:
print(df.dtypes)


## Data Structure & Statistical Summary

- The dataset contains structured demographic counts across dates, states, and districts.
- Numerical columns show wide variance, indicating uneven enrollment patterns across regions.
- No structural inconsistencies were observed in data types.


In [ ]:
# Check for missing values
print(df.isnull().sum())

## Missing Values Assessment

An evaluation of missing values across all columns revealed **no missing entries** in the dataset.
This confirms data completeness and eliminates the need for imputation or row removal,
allowing the analysis to proceed without additional data cleaning steps.


In [ ]:
print(df.duplicated().sum())

In [ ]:
df[df.duplicated()].head()

## State Name Standardization

Certain Union Territories appeared multiple times due to spelling and naming variations 
(e.g., "&" vs "and", legacy administrative names).

These are **not duplicate records**, but **semantic inconsistencies** that can lead to 
incorrect aggregation during state- and district-level analysis.

State names were standardized using official post-merger administrative nomenclature to:
- Ensure accurate aggregation
- Prevent double counting
- Improve analytical reliability


In [ ]:
# ---------------------------------------------------------
# STEP 1: Standardize 'df' BEFORE any Aggregation
# ---------------------------------------------------------

# First, fix case and whitespace on the raw dataframe
df["state"] = df["state"].str.strip().str.title()

state_standardization = {
    "West Bangal": "West Bengal", "West  Bengal": "West Bengal", "Westbengal": "West Bengal",
    "Orissa": "Odisha",
    "Chhatisgarh": "Chhattisgarh",
    "Uttaranchal": "Uttarakhand",
    "Tamilnadu": "Tamil Nadu",
    "Jammu & Kashmir": "Jammu And Kashmir", "Jammu and Kashmir": "Jammu And Kashmir",
    "Andaman & Nicobar Islands": "Andaman And Nicobar Islands",
    "Andaman and Nicobar Islands": "Andaman And Nicobar Islands",
    "Daman & Diu": "Dadra And Nagar Haveli And Daman And Diu",
    "Daman And Diu": "Dadra And Nagar Haveli And Daman And Diu",
    "Dadra & Nagar Haveli": "Dadra And Nagar Haveli And Daman And Diu",
    "Dadra And Nagar Haveli": "Dadra And Nagar Haveli And Daman And Diu"
}

df["state"] = df["state"].replace(state_standardization)

# ---------------------------------------------------------
# STEP 2: Now Aggregate the Cleaned Data
# ---------------------------------------------------------
# Since 'state' is now clean, this groupby will merge all variations correctly
district_level = (
    df
    .drop(columns=['pincode'])
    .groupby(['date', 'state', 'district'], as_index=False)
    .sum()
)

# Now you can safely create clean_df from district_level
clean_df = district_level.copy()

In [ ]:
df["state"].value_counts().loc[
    df["state"].value_counts().index.str.contains(
        "Bengal|Kashmir|Odisha|Puducherry|Dadra|Daman",
        case=False
    )
]


In [ ]:

# 1. Clean raw df
df["state"] = df["state"].str.strip().str.title()
df["state"] = df["state"].replace(state_standardization)

# 2. Aggregate
district_level = (
    df.drop(columns=['pincode'])
    .groupby(['date', 'state', 'district'], as_index=False)
    .sum()
)

## Duplicate Records Handling

The dataset contained multiple records for the same administrative units
due to repeated entries at the pincode level across dates.

Rather than removing duplicates outright, records were **aggregated at the
district level** using summation for numerical demographic fields.
This approach preserves total enrollment counts while eliminating redundancy,
ensuring accurate district-level analysis.


In [ ]:
district_level = (
    df
    .drop(columns=['pincode'])
    .groupby(['date', 'state', 'district'], as_index=False)
    .sum()
)


## District-Level Aggregation

Data was aggregated at the district level to eliminate pincode-level noise and enable meaningful regional comparison and anomaly detection.


In [ ]:
district_level.head()


In [ ]:
district_level.describe()


In [ ]:
district_level.sort_values('demo_age_17_', ascending=False).head(10)


In [ ]:
district_level.sort_values('demo_age_5_17').head(10)


## Feature Engineering
Derived metrics and flags are created to enhance analytical depth, including demographic ratios and reporting issue indicators.


In [ ]:
# 3. Create the missing flag that the function needs
district_level['reporting_issue_flag'] = (
    (district_level['demo_age_5_17'] == 0) & 
    (district_level['demo_age_17_'] < 50)
)

# 4. Create the ratio column
district_level['child_ratio'] = (
    district_level['demo_age_5_17'] / 
    (district_level['demo_age_17_'] + 1)
)

# 5. Convert date and extract year/month
district_level['date'] = pd.to_datetime(district_level['date'], dayfirst=True)
district_level['year'] = district_level['date'].dt.year
district_level['month'] = district_level['date'].dt.month

In [ ]:
district_level['reporting_issue_flag'].value_counts()


In [ ]:
final_df = district_level.copy()


In [ ]:
final_df['date'] = pd.to_datetime(final_df['date'], dayfirst=True)
final_df['year'] = final_df['date'].dt.year
final_df['month'] = final_df['date'].dt.month


In [ ]:
final_df['child_ratio'] = (
    final_df['demo_age_5_17'] /
    (final_df['demo_age_17_'] + 1)
)


In [ ]:
final_df['enrollment_level'] = pd.cut(
    final_df['demo_age_5_17'],
    bins=[-1, 10, 100, 1000, 10000],
    labels=['Very Low', 'Low', 'Medium', 'High']
)


In [ ]:
clean_df = final_df[~final_df['reporting_issue_flag']]


In [ ]:


# 1. Select the numerical columns for correlation
# We include the raw counts and the engineered child_ratio
corr_data = clean_df[['demo_age_5_17', 'demo_age_17_', 'child_ratio']]

# 2. Calculate the correlation matrix
correlation_matrix = corr_data.corr()

# 3. Visualize using a Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0, fmt=".2f")
plt.title('Correlation Matrix: Child vs Adult Enrollment Activity')
plt.show()



# Insight: 
A high correlation between age groups suggests uniform administrative activity, 
while a low correlation with child_ratio highlights where specific age-based drives are occurring.

## National Enrollment Trends
This section analyzes Aadhaar enrollment trends at the national level to understand overall system behavior over time.

A consistent national trend highlights periodic enrollment surges, potentially linked to policy initiatives, enrollment drives, or life-event registrations.



In [ ]:
national_trend = (
    clean_df
    .groupby('date', as_index=False)
    [['demo_age_5_17', 'demo_age_17_']]
    .sum()
)



In [ ]:
national_trend.head()
national_trend.dtypes


In [ ]:
clean_df['year_month'] = clean_df['date'].dt.to_period('M').astype(str)


In [ ]:
national_trend = (
    clean_df
    .groupby('year_month', as_index=False)[
        ['demo_age_5_17', 'demo_age_17_']
    ]
    .sum()
)


In [ ]:
plt.figure(figsize=(13,6))

plt.plot(
    national_trend['year_month'],
    national_trend['demo_age_17_'],
    marker='o',
    linewidth=2.5,
    label='Age 17+ Updates'
)

plt.plot(
    national_trend['year_month'],
    national_trend['demo_age_5_17'],
    marker='s',
    linewidth=2.5,
    label='Age 5–17 Updates'
)

plt.title(
    "National Monthly Trend in Aadhaar Demographic Updates (2025)",
    fontsize=14,
    pad=12
)

plt.xlabel("Month", fontsize=11)
plt.ylabel("Number of Updates", fontsize=11)

plt.xticks(rotation=45)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## National Aadhaar Activity Trends (Combined View)

This chart presents a unified national-level view of Aadhaar activity over time by 
combining enrolments, demographic updates, and biometric updates into a single timeline.

### Key Insights:
- Enables direct comparison between new enrolments and update activity.
- Highlights periods of increased system load or policy-driven surges.
- Reveals whether Aadhaar growth is driven more by new registrations or by updates.

This consolidated visualization avoids redundancy and provides a clearer 
macro-level understanding of Aadhaar system dynamics across India.


In [ ]:
national_trend.shape


### National Child-to-Adult Update Ratio

The child ratio reflects the balance between updates for children and adults.
Sudden changes may indicate school admissions, documentation drives, or policy effects.


In [ ]:
monthly_child_ratio = (
    clean_df
    .groupby("month", as_index=False)[
        ["demo_age_5_17", "demo_age_17_"]
    ]
    .sum()
)

monthly_child_ratio["child_ratio"] = (
    monthly_child_ratio["demo_age_5_17"] /
    (monthly_child_ratio["demo_age_17_"] + 1)
)

plt.figure(figsize=(10,4))
plt.plot(
    monthly_child_ratio["month"],
    monthly_child_ratio["child_ratio"],
    marker="o"
)

plt.title("Monthly National Child-to-Adult Update Ratio (2025)")
plt.xlabel("Month")
plt.ylabel("Child Ratio")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## State-Level Enrollment Trends
State-wise aggregation highlights regional disparities and enrollment concentration across India.



In [ ]:
state_total = (
    clean_df
    .groupby("state")[["demo_age_5_17", "demo_age_17_"]]
    .sum()
)
state_total["total"] = (
    state_total["demo_age_5_17"] +
    state_total["demo_age_17_"]
)

top_states = state_total.sort_values("total", ascending=False).head(10)

plt.figure(figsize=(8,4))
plt.barh(top_states.index, top_states["total"])
plt.title("Top 10 States Contributing to National Demographic Updates")
plt.xlabel("Total Updates")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Concentration of National Demographic Activity

This chart highlights the states contributing the highest share of Aadhaar
demographic updates, revealing regional concentration and operational load.




📌 National Aadhaar Activity — Integrated Insights

Together, these national-level analyses provide a holistic view of Aadhaar system dynamics in India. The combined activity trend highlights the overall evolution of enrolments, biometric updates, and demographic updates, reflecting system-wide demand and operational load over time. Complementing this, the child-to-adult update ratio reveals underlying demographic drivers, indicating periods where lifecycle-driven updates (such as child growth transitions) contribute disproportionately to Aadhaar modifications. Finally, the state-wise contribution analysis uncovers strong regional concentration, showing that a small set of states accounts for a significant share of national demographic updates. Viewed collectively, these insights move beyond raw counts to explain how, who, and where Aadhaar activity is generated—offering actionable context for capacity planning, targeted outreach, and policy prioritization

In [ ]:
state_trend = (
    clean_df
    .groupby(['year', 'state'], as_index=False)
    [['demo_age_5_17', 'demo_age_17_']]
    .sum()
)


### States with High Child Update Ratios

A high child ratio may indicate school admissions, birth registration cycles,
or targeted documentation drives focused on younger populations.


In [ ]:
state_trend["child_ratio"] = (
    state_trend["demo_age_5_17"] /
    (state_trend["demo_age_17_"] + 1)
)

top_child_states = (
    state_trend
    .sort_values("child_ratio", ascending=False)
    .head(10)
)

plt.figure(figsize=(8,4))
plt.barh(top_child_states["state"], top_child_states["child_ratio"])
plt.title("Top States by Child-to-Adult Update Ratio")
plt.xlabel("Child Ratio")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
state_trend.head()
state_trend.dtypes


## State-wise Enrollment Trends

Significant inter-state variation was observed.  
High-population states exhibit sustained enrollment activity, while smaller states show sporadic spikes, indicating uneven access or reporting cycles.


In [ ]:
state_trend.sort_values('demo_age_5_17', ascending=False).head(10)


## Age Group Dynamics

- Adult enrollment (17+) dominates Aadhaar updates.
- Child enrollment patterns highlight school-age registration drives and demographic growth.
- The ratio between age groups provides insight into population transition stages.


In [ ]:
state_trend['child_ratio'] = (
    state_trend['demo_age_5_17'] /
    (state_trend['demo_age_17_'] + 1)
)


In [ ]:
state_trend.sort_values('child_ratio', ascending=False).head(10)


In [ ]:
state_trend['child_ratio'] = (
    state_trend['demo_age_5_17'] /
    (state_trend['demo_age_17_'] + 1)
)


In [ ]:
state_trend.sort_values('child_ratio').head(10)


In [ ]:
state_trend.sort_values('child_ratio', ascending=False).head(10)


In [ ]:


top_states = state_trend.sort_values('child_ratio', ascending=False).head(10)

plt.figure(figsize=(8,4))
plt.barh(top_states['state'], top_states['child_ratio'])
plt.title('Top States by Child Enrollment Ratio (2025)')
plt.xlabel('Child Ratio')
plt.ylabel('State')
plt.show()


In [ ]:
key_states = ["Uttar Pradesh", "Maharashtra"]

monthly_state = (
    clean_df[clean_df["state"].isin(key_states)]
    .groupby(["month", "state"], as_index=False)[
        ["demo_age_5_17", "demo_age_17_"]
    ]
    .sum()
)

monthly_state["total_updates"] = (
    monthly_state["demo_age_5_17"] +
    monthly_state["demo_age_17_"]
)

plt.figure(figsize=(10,4))
sns.lineplot(
    data=monthly_state,
    x="month",
    y="total_updates",
    hue="state",
    marker="o"
)

plt.title("Monthly Demographic Update Trends in Key States (2025)")
plt.xlabel("Month")
plt.ylabel("Total Updates")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Monthly Trends in High-Impact States

Examining intra-year patterns for major states helps identify seasonal peaks,
operational load, and potential policy-driven effects.


In [ ]:
state_month = (
    clean_df
    .groupby(["state", "month"], as_index=False)[
        ["demo_age_5_17", "demo_age_17_"]
    ]
    .sum()
)

state_month["total_updates"] = (
    state_month["demo_age_5_17"] +
    state_month["demo_age_17_"]
)

pivot = state_month.pivot(
    index="state",
    columns="month",
    values="total_updates"
)

plt.figure(figsize=(12,8))
sns.heatmap(pivot, cmap="YlOrRd")
plt.title("State-wise Monthly Intensity of Demographic Updates (2025)")
plt.xlabel("Month")
plt.ylabel("State")
plt.tight_layout()
plt.show()


### Seasonal and Regional Intensity of Demographic Updates

The heatmap reveals how demographic update activity varies across states and months,
highlighting seasonal patterns and region-specific demand surges.


## Anomaly Detection and Severity Classification
Anomalies are identified using demographic ratios and reporting flags. Records are classified into severity levels to prioritize intervention.


In [ ]:
district_level = (
    clean_df
    .groupby(["state", "district"], as_index=False)[
        ["demo_age_5_17", "demo_age_17_"]
    ]
    .sum()
)

district_level["total_updates"] = (
    district_level["demo_age_5_17"] +
    district_level["demo_age_17_"]
)

top_outliers = district_level.sort_values(
    "total_updates", ascending=False
).head(10)

plt.figure(figsize=(9,4))
plt.barh(
    top_outliers["district"] + ", " + top_outliers["state"],
    top_outliers["total_updates"]
)
plt.title("Top 10 Districts with Unusually High Demographic Updates")
plt.xlabel("Total Updates")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### High-Volume District Anomalies

Districts with exceptionally high update counts may reflect dense populations,
migration hubs, or intensified documentation drives. These areas warrant closer
operational monitoring.


In [ ]:
low_activity = district_level[
    district_level["total_updates"] < 10
].sort_values("total_updates")

plt.figure(figsize=(9,4))
plt.barh(
    low_activity["district"].head(10) + ", " + low_activity["state"].head(10),
    low_activity["total_updates"].head(10)
)
plt.title("Districts with Near-Zero Demographic Updates")
plt.xlabel("Total Updates")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Low-Activity Districts

Near-zero update activity may indicate data reporting gaps, limited Aadhaar access,
or temporary service disruptions. Such districts require administrative follow-up.


In [ ]:
def anomaly_severity(row):
    if row['demo_age_17_'] == 0:
        return 'Critical'
    elif row['child_ratio'] < 0.01 or row['child_ratio'] > 0.5:
        return 'High'
    elif row['reporting_issue_flag']:
        return 'Medium'
    else:
        return 'Normal'

clean_df['anomaly_severity'] = clean_df.apply(anomaly_severity, axis=1)


### Anomaly Severity Distribution
The distribution of anomaly severity levels provides an overview of data reliability and operational stability.


In [ ]:
clean_df['anomaly_severity'].value_counts()


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=district_level["total_updates"])
plt.title("Distribution of District-Level Demographic Updates")
plt.xlabel("Total Updates")
plt.tight_layout()
plt.show()


### Distribution of District-Level Activity

The boxplot highlights the spread of district-level updates and helps distinguish
true outliers from normal variation.


## Key Insights
- National Aadhaar enrollment is dominated by the adult (17+) population, indicating near-saturation.
- Significant variation exists in child-to-adult enrollment ratios across districts.
- Approximately 3.5% of records exhibit high or critical anomalies, including zero enrollment values.
- Anomalies are geographically concentrated, suggesting localized operational issues.


## Proposed Solutions and System Improvements
- Implement automated anomaly alerts for zero or extreme enrollment values.
- Use severity-based classification to prioritize district-level interventions.
- Monitor demographic ratios as leading indicators of outreach effectiveness.
- Deploy interactive dashboards for real-time monitoring and decision support.
- Conduct targeted audits in high-risk districts.


##  District Risk Scoring Framework

To make the analysis actionable for policymakers, we convert anomaly detection into a **risk-based prioritization framework**.

Each district is classified into one of three categories:
- **Normal** → No immediate action required
- **High Risk** → Needs monitoring and field verification
- **Critical Risk** → Requires urgent administrative intervention

This allows authorities to allocate resources efficiently instead of reacting uniformly across all regions.


## District-Level Anomaly Severity Classification

To prioritize regions requiring intervention, districts are categorized based on
child enrollment ratios and reporting inconsistencies into severity tiers.


In [ ]:
district_level['child_ratio'] = (
    district_level['demo_age_5_17'] /
    (district_level['demo_age_17_'] + 1)
)


In [ ]:
def classify_severity(row):
    if row['reporting_issue_flag'] and row['child_ratio'] < 0.1:
        return 'Critical'
    elif row['reporting_issue_flag'] or row['child_ratio'] < 0.2:
        return 'High'
    else:
        return 'Normal'

district_level['anomaly_severity'] = district_level.apply(
    classify_severity, axis=1
)


In [ ]:
district_level['anomaly_severity'].value_counts()


In [ ]:
# Assign numerical risk score
risk_map = {
    "Normal": 1,
    "High": 2,
    "Critical": 3
}

district_level["risk_score"] = district_level["anomaly_severity"].map(risk_map)


In [ ]:
# Top 10 highest risk districts
top_risk_districts = district_level.sort_values(
    by="risk_score", ascending=False
).head(10)

top_risk_districts[[
    "state", "district", 
    "demo_age_5_17", 
    "demo_age_17_", 
    "child_ratio", 
    "anomaly_severity"
]]


### 🔍 Key Observation

A small subset of districts contributes disproportionately to enrollment anomalies.
These districts consistently show:
- Extremely low child enrollment
- High deviation from national patterns
- Repeated reporting inconsistencies

Targeting just the **top 10 high-risk districts** could resolve a significant share of Aadhaar enrollment issues nationwide.


##  Policy Recommendations & Solutions

Based on the risk analysis, the following actions are recommended:

### 🔴 Critical Risk Districts
- Immediate audit of enrollment centers
- Temporary mobile Aadhaar units
- Verification of reporting pipelines

### 🟠 High Risk Districts
- Monthly monitoring dashboards
- Awareness campaigns targeting parents
- Local administrative reviews

### 🟢 Normal Districts
- Continue standard operations
- Use as benchmarks for best practices

This tiered approach ensures optimal use of government resources while improving enrollment coverage.


## Aadhaar Kendra Performance Index (District-Level Proxy)

To evaluate the operational effectiveness of Aadhaar Kendras, a district-level
performance proxy is derived using enrollment consistency and anomaly patterns.

### Rationale
- Districts with stable enrollment trends indicate efficient service delivery.
- Sudden spikes or drops may reflect operational bottlenecks, reporting issues,
  or temporary access disruptions.
- Repeated anomalies can signal infrastructure or staffing gaps.

### Interpretation
- **High Performance**: Consistent enrollment, low anomalies
- **Moderate Performance**: Minor fluctuations
- **Low Performance**: High anomaly frequency or extreme deviations

This index can assist UIDAI and state authorities in identifying districts
that may require targeted operational improvements.


## Digital Divide Index (Demographic Enrollment Imbalance)

A Digital Divide Index is inferred using the ratio between child (5–17 years)
and adult (17+ years) Aadhaar enrollments.

### Insight Logic
- Delayed child enrollment may indicate lack of awareness, accessibility,
  or socio-economic barriers.
- Balanced child-to-adult ratios suggest stronger digital inclusion.
- Persistent gaps highlight regions requiring focused outreach programs.

### Key Observations
- Urban districts tend to show earlier stabilization in child enrollment.
- Several rural or remote districts exhibit delayed or irregular child registration.

This indicator helps policymakers identify regions where digital inclusion
efforts need strengthening.


## Policy Impact & Practical Applications

The insights derived from this analysis can support:

- **Targeted Outreach**: Identifying districts with delayed child enrollment
- **Operational Planning**: Improving Aadhaar Kendra efficiency
- **Data Governance**: Early detection of reporting inconsistencies
- **Digital Inclusion**: Measuring access gaps across regions

The framework enables evidence-based decision-making
for UIDAI and associated government agencies.


## Future Enhancements

With the integration of additional datasets such as migration records,
disaster reports, or telecom metadata, this framework can be extended to:

- Migration Intelligence Engine
- Disaster & Crisis Enrollment Signals
- Real-time Aadhaar Kendra Performance Monitoring
- Predictive Enrollment Forecasting

These extensions would further strengthen Aadhaar’s role
as a backbone for inclusive governance.


## Conclusion
This analysis demonstrates how descriptive analytics, demographic indicators, and anomaly detection can support data-driven decision-making in large-scale public systems such as Aadhaar enrollment. The proposed solution framework emphasizes interpretability, scalability, and policy relevance.


## Data Export for Visualization
The final cleaned and enriched dataset is exported for dashboard development and stakeholder consumption.


In [ ]:
final_df.to_csv('aadhaar_final_analysis.csv', index=False)


In [ ]:
from IPython.display import FileLink
FileLink("aadhaar_final_analysis.csv")
